In [60]:
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score,classification_report
import pandas as pd
import seaborn as sns

In [61]:
df = pd.read_excel('intrusion_dataset_realistic_50000.xlsx')
df.head()

,duration,protocol_type,src_bytes,dst_bytes,failed_logins,count,serror_rate,dst_host_count,label
0,109,tcp,12252,5218,6,18,0.719,124,DoS
1,86,tcp,6257,6782,7,74,0.401,225,BruteForce
2,25,tcp,3325,14756,0,69,0.632,196,Normal
3,84,tcp,6740,13871,0,62,0.849,106,BruteForce
4,92,udp,12129,4309,6,28,0.235,77,Normal


In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   duration        50000 non-null  int64  
 1   protocol_type   50000 non-null  object 
 2   src_bytes       50000 non-null  int64  
 3   dst_bytes       50000 non-null  int64  
 4   failed_logins   50000 non-null  int64  
 5   count           50000 non-null  int64  
 6   serror_rate     50000 non-null  float64
 7   dst_host_count  50000 non-null  int64  
 8   label           50000 non-null  object 
dtypes: float64(1), int64(6), object(2)
memory usage: 3.4+ MB


In [63]:
x = df.drop(columns=['label'],axis=1)
y = df['label']

In [64]:
encoder = OneHotEncoder(sparse_output=False,handle_unknown='ignore')

In [65]:
encod = encoder.fit_transform(x[['protocol_type']])
encod

array([[0., 1., 0.],
       [0., 1., 0.],
       [0., 1., 0.],
       ...,
       [0., 1., 0.],
       [0., 1., 0.],
       [0., 1., 0.]])

In [66]:
x[encoder.get_feature_names_out()] = encod 

In [67]:
x.drop(columns=['protocol_type'],axis=1,inplace=True)
x

,duration,src_bytes,dst_bytes,failed_logins,count,serror_rate,dst_host_count,protocol_type_icmp,protocol_type_tcp,protocol_type_udp
0,109,12252,5218,6,18,0.719,124,0.0,1.0,0.0
1,86,6257,6782,7,74,0.401,225,0.0,1.0,0.0
2,25,3325,14756,0,69,0.632,196,0.0,1.0,0.0
3,84,6740,13871,0,62,0.849,106,0.0,1.0,0.0
4,92,12129,4309,6,28,0.235,77,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
49995,16,3664,4989,7,70,0.992,68,0.0,1.0,0.0
49996,111,4403,4308,0,46,0.122,12,0.0,1.0,0.0
49997,19,9781,14969,7,37,0.294,71,0.0,1.0,0.0
49998,60,11729,14453,1,26,0.535,49,0.0,1.0,0.0


In [68]:
xtrain, xtest, ytrain, ytest = train_test_split(x,y,train_size=0.8,random_state=42)

In [69]:
model = RandomForestClassifier()

In [70]:
model.fit(xtrain,ytrain)

RandomForestClassifier()

In [71]:
y_pred = model.predict(xtrain)
y_pred

array(['Normal', 'Normal', 'Normal', ..., 'Normal', 'Normal', 'Probe'],
      dtype=object)

In [73]:
print(classification_report(ytrain,y_pred))

              precision    recall  f1-score   support

  BruteForce       1.00      1.00      1.00      4777
         DoS       1.00      1.00      1.00      2957
      Normal       1.00      1.00      1.00     23183
       Probe       1.00      1.00      1.00      9083

    accuracy                           1.00     40000
   macro avg       1.00      1.00      1.00     40000
weighted avg       1.00      1.00      1.00     40000



In [74]:
y_pred_test = model.predict(xtest)
y_pred_test

array(['Normal', 'BruteForce', 'Normal', ..., 'Probe', 'Probe', 'Normal'],
      dtype=object)

In [75]:
print(classification_report(ytest,y_pred_test))

              precision    recall  f1-score   support

  BruteForce       0.51      0.55      0.53      1181
         DoS       0.80      0.71      0.76       721
      Normal       0.88      0.88      0.88      5829
       Probe       0.51      0.51      0.51      2269

    accuracy                           0.74     10000
   macro avg       0.67      0.66      0.67     10000
weighted avg       0.74      0.74      0.74     10000

